# Análisis Exploratorio de Datos (EDA) Profesional

## Objetivos del Modelo de Machine Learning
En este proyecto, hemos diseñado un pipeline de datos robusto con el objetivo de entrenar modelos de Inteligencia Artificial para dos tareas principales:

1. **Regresión:** Queremos predecir el **monto total de venta** (`monto_total_venta`) basado en el comportamiento del cliente, canales de compra y stock de productos. *Nota: Hemos excluido variables como `cantidad` y `precio_unitario` para evitar Fuga de Datos (Data Leakage).* 
2. **Clasificación:** Buscamos clasificar a los clientes en su respectivo **segmento** (ej. Premium, VIP, Nuevo) basado en su histórico de transacciones.
3. **Clustering:** Segmentación no supervisada para descubrir perfiles ocultos de clientes.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración visual profesional
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Carga de Datos Crudos
Cargaremos las ventas directamente desde la capa `01_raw` para analizar su estado original.

In [ ]:
df_ventas = pd.read_csv('../data/01_raw/ventas.csv')
df_ventas.head()

## 2. Análisis de Valores Extremos (Outliers) en Precios
Antes de modelar, es crítico entender la distribución de las variables numéricas. En un entorno real, es común encontrar transacciones anómalas (precios muy altos). 

**Decisión Arquitectónica:** En nuestro pipeline de Kedro, implementamos **Winzorización** (percentiles 5 y 95) en lugar de eliminación por IQR. Esto nos permite mitigar el impacto de los outliers *sin perder filas*, conservando datos valiosos de clientes que, aunque gastaron mucho, son reales.

In [ ]:
# Convertimos a numérico para evitar errores si hay strings
df_ventas['precio_unitario'] = pd.to_numeric(df_ventas['precio_unitario'], errors='coerce')

fig, ax = plt.subplots(1, 2, figsize=(15, 5))

sns.boxplot(x=df_ventas['precio_unitario'], ax=ax[0], color="#4C72B0")
ax[0].set_title('Distribución con Outliers Severos', fontweight='bold')

# Simulamos la Winzorización que hace el pipeline
lower = df_ventas['precio_unitario'].quantile(0.05)
upper = df_ventas['precio_unitario'].quantile(0.95)
precio_winsorized = df_ventas['precio_unitario'].clip(lower=lower, upper=upper)

sns.boxplot(x=precio_winsorized, ax=ax[1], color="#55A868")
ax[1].set_title('Distribución tras Winzorización (P5-P95)', fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Distribución de Canales de Venta
Para el modelo de clasificación de segmentos, entender los canales por los que interactúa el cliente es fundamental. Nuestro pipeline convierte estas variables mediante One-Hot Encoding (`StandardScaler` se encarga de las numéricas).

In [ ]:
# Limpieza rápida para el gráfico (similar a lo que hace el pipeline)
df_ventas['canal_venta'] = df_ventas['canal_venta'].astype(str).str.strip().str.title()
df_ventas = df_ventas[df_ventas['canal_venta'] != 'Nan']

plt.figure(figsize=(8, 5))
sns.countplot(data=df_ventas, x='canal_venta', order=df_ventas['canal_venta'].value_counts().index, palette='viridis')
plt.title('Frecuencia por Canal de Venta', fontweight='bold')
plt.ylabel('Cantidad de Transacciones')
plt.xlabel('Canal')
plt.show()